# Lab 12: LSTM Model for Time-Series Forecasting

**Student Name:** Malik Awais Bashir  
**Registration Number:** 22JZELE0474  
**Course:** Machine Learning Lab  
**Supervisor:** Engr. Irshad Ullah  
**University:** UET Peshawar - Nowshera Campus  

## Goal of this Lab
* Set up the workspace and import the required Scikit-learn, TensorFlow/Keras, and time-series processing libraries.
* Design an LSTM (Long Short-Term Memory) network for sequential data forecasting.
* Configure training settings, including the optimizer, model checkpoints, and callback functions.
* Load the training, validation, and testing datasets from CSV files for forecasting tasks.
* Assess the forecasting performance of the LSTM model using MAE, MSE, RMSE, R² score, and explained variance metrics.


## Section 1: Working Directory and Library Imports
This section sets the Lab 12 working directory and imports Scikit-learn metrics, TensorFlow/Keras layers, callbacks, optimizers, and custom time-series utility functions.


In [72]:
import os
os.chdir(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12')

In [73]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score, r2_score
from timeseires.utils.to_split import to_split
from timeseires.utils.multivariate_multi_step import multivariate_multi_step
from timeseires.utils.multivariate_single_step import multivariate_single_step
from timeseires.utils.univariate_multi_step import univariate_multi_step
from timeseires.utils.univariate_single_step import univariate_single_step
from timeseires.utils.CosineAnnealingLRS import CosineAnnealingLRS
from timeseires.callbacks.EpochCheckpoint import EpochCheckpoint
from tensorflow.keras.callbacks import ModelCheckpoint
from timeseires.callbacks.TrainingMonitor import TrainingMonitor
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import LSTM, Bidirectional, Add
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv1D,TimeDistributed
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten,MaxPooling1D,Concatenate,AveragePooling1D, GlobalMaxPooling1D, Input
from tensorflow.keras.models import Sequential,Model
import pandas as pd
import time, pickle
import numpy as np
import tensorflow.keras.backend as K
import tensorflow
from tensorflow.keras.layers import Input, Reshape, Lambda
from tensorflow.keras.layers import Layer, Flatten, LeakyReLU, concatenate, Dense
from tensorflow.keras.regularizers import l2
import glob
import h5py
import matplotlib.pyplot as plt
from keras.callbacks import Callback

In [74]:
#lookback = 24
model = None
start_epoch = 0
time_steps=24
num_features=21

In [75]:
def create_lstm():
    input_data = Input(shape=(time_steps, num_features))
    lstm_layer1 = LSTM(8, return_sequences=True)(input_data)
    lstm_layer2 = LSTM(20)(lstm_layer1)
    x = Flatten()(lstm_layer2)
    output_data = Dense(1)(x)
    model = Model(input_data, output_data)
    return model

## Section 2: Model Parameters and LSTM Architecture
The following cells define time steps, feature count, and the LSTM model architecture used for sequential forecasting.


In [76]:
model1 = create_lstm()
model1.summary()

Model: "functional_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_10 (InputLayer)     │ (None, 24, 21)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_20 (LSTM)                  │ (None, 24, 8)          │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_21 (LSTM)                  │ (None, 20)             │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_10 (Flatten)            │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,301 (12.89 KB)

 Trainable params: 3,301 (12.89 KB)

 Non-trainable params: 0 (0.00 B)

In [77]:
tensorflow.keras.utils.plot_model(model1 )

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [78]:
checkpoints = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\\E1-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
OUTPUT_PATH = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12'
FIG_PATH = os.path.sep.join([OUTPUT_PATH,"\history.png"])
JSON_PATH = os.path.sep.join([OUTPUT_PATH,"\history.json"])

In [79]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]

## Section 3: Checkpoints, Callbacks, and Training Configuration
This section prepares checkpoint paths, output directories, training monitor callbacks, optimizer settings, and model compilation.


In [80]:
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model =create_lstm()
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model))
    model = load_model(model)

    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.lr)))
    K.set_value(model.optimizer.lr, 1e-4)
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.lr)))

[INFO] compiling model...


In [81]:
path_dataset = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12'

path_tr = os.path.join(path_dataset, 'AEP_train.csv')
path_v = os.path.join(path_dataset, 'AEP_validation.csv')
path_te = os.path.join(path_dataset, 'AEP_test.csv')
path_hourly = os.path.join(path_dataset, 'AEP_hourly.csv')

for path in [path_tr, path_v, path_te, path_hourly]:
    print(path, "=>", os.path.exists(path))

df_tr = pd.read_csv(path_tr)
train_set = df_tr.values

df_v = pd.read_csv(path_v)
validation_set = df_v.values

df_te = pd.read_csv(path_te)
test_set = df_te.values

df_hourly = pd.read_csv(path_hourly)

train_set.shape, validation_set.shape, test_set.shape, df_hourly.shape

C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\AEP_train.csv => True
C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\AEP_validation.csv => True
C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\AEP_test.csv => True
C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\AEP_hourly.csv => True


((84907, 21), (24259, 21), (12130, 21), (121273, 2))

In [82]:
time_steps=24
num_features=21

In [83]:
start = time.time()

train_X, train_y = univariate_multi_step(
    train_set,
    time_steps,
    target_col=0,
    target_len=1
)

validation_X, validation_y = univariate_multi_step(
    validation_set,
    time_steps,
    target_col=0,
    target_len=1
)

test_X, test_y = univariate_multi_step(
    test_set,
    time_steps,
    target_col=0,
    target_len=1
)

print('Time Consumed', time.time() - start, "sec")

Time Consumed 1.0603744983673096 sec


## Section 4: Dataset Loading and Forecast Preparation
The following cells load train, validation, and test CSV files, then prepare the data for LSTM forecasting input/output format.


In [84]:
epochs = 2
verbose = 1
batch_size = 32

History = model.fit(
    train_X,
    train_y,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(validation_X, validation_y),
    callbacks=callbacks,
    verbose=verbose
)

Epoch 1/2
2652/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0699 - mae: 0.0699 - mape: 1208.8188
Epoch 1: val_loss improved from None to 0.01992, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\\E1-cp-0001-loss0.02.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 73s 25ms/step - loss: 0.0425 - mae: 0.0425 - mape: 1323.4902 - val_loss: 0.0199 - val_mae: 0.0199 - val_mape: 8.3445
Epoch 2/2
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0192 - mae: 0.0192 - mape: 281.8764
Epoch 2: val_loss improved from 0.01992 to 0.01435, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\\E1-cp-0002-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 64s 24ms/step - loss: 0.0172 - mae: 0.0172 - mape: 180.8920 - val_loss: 0.0143 - val_mae: 0.0143 - val_mape: 5.8791


In [85]:
import os
import pickle

path_dataset = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12'
path_scaler = os.path.join(path_dataset, 'AEP_Scaler.pkl')

print(os.path.exists(path_scaler))

with open(path_scaler, 'rb') as f:
    scaler = pickle.load(f)

True


c:\Users\hp\anaconda3\envs\dsp\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [86]:
model_path = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\\E1-cp-0001-loss0.02.h5'

model = load_model(model_path, compile=False)

y_pred_scaled = model.predict(test_X)

print("y_pred_scaled.shape =", y_pred_scaled.shape)
print("test_y.shape =", test_y.shape)

def inverse_target(scaled_values, scaler, target_col=0, num_features=21):
    scaled_values = scaled_values.reshape(-1)

    dummy = np.zeros((scaled_values.shape[0], num_features))
    dummy[:, target_col] = scaled_values

    unscaled = scaler.inverse_transform(dummy)

    return unscaled[:, target_col].reshape(-1, 1)

y_pred = inverse_target(
    y_pred_scaled,
    scaler,
    target_col=0,
    num_features=21
)

y_test_unscaled = inverse_target(
    test_y,
    scaler,
    target_col=0,
    num_features=21
)

# Mean Absolute Error (MAE)
MAE = np.mean(np.abs(y_pred - y_test_unscaled))
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(np.abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.mean(np.square(y_pred - y_test_unscaled))
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squared Error (RMSE)
RMSE = np.sqrt(MSE)
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median(np.abs((y_test_unscaled - y_pred) / y_test_unscaled)) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape = ', y_test_unscaled.shape)
print('y_pred.shape = ', y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step
y_pred_scaled.shape = (12105, 1)
test_y.shape = (12105, 1)
Mean Absolute Error (MAE): 320.39
Median Absolute Error (MedAE): 258.84
Mean Squared Error (MSE): 169821.59
Root Mean Squared Error (RMSE): 412.09
Mean Absolute Percentage Error (MAPE): 2.17 %
Median Absolute Percentage Error (MDAPE): 1.8 %


y_test_unscaled.shape =  (12105, 1)
y_pred.shape =  (12105, 1)


In [87]:
checkpoints = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\E2-cp-{epoch:04d}-loss{val_loss:.2f}.h5'
model=r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\\E1-cp-0001-loss0.02.h5'
start_epoch= 34

## Section 5: Prediction and Model Evaluation
The final cells generate predictions and calculate regression metrics to evaluate the forecasting model performance.


In [88]:
# construct the callback to save only the *best* model to disk
# based on the validation loss
EpochCheckpoint1 = ModelCheckpoint(checkpoints,
                             monitor="val_loss",
                             save_best_only=True, 
                             verbose=1)
TrainingMonitor1=TrainingMonitor(FIG_PATH, jsonPath=JSON_PATH, startAt=start_epoch)

# construct the set of callbacks
callbacks = [EpochCheckpoint1,TrainingMonitor1]
model_load = r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\\E1-cp-0001-loss0.02.h5'
# if there is no specific model checkpoint supplied, then initialize
# the network and compile the model
if model is None:
    print("[INFO] compiling model...")
    model = PC.build(time_steps=24, num_features=21, reg=0.0005)
    opt = Adam(1e-3)
    model.compile(loss= 'mae', optimizer=opt, metrics=["mae", "mape"])
# otherwise, load the checkpoint from disk
else:
    print("[INFO] loading {}...".format(model_load))
    model = load_model(model_load, compile=False)

    opt = Adam(learning_rate=1e-4)
    model.compile(
        loss="mae",
        optimizer=opt,
        metrics=["mae", "mape"]
    )


    # update the learning rate
    print("[INFO] old learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))
    print("[INFO] new learning rate: {}".format(K.get_value(model.optimizer.learning_rate)))

[INFO] loading C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\\E1-cp-0001-loss0.02.h5...
[INFO] old learning rate: 9.999999747378752e-05
[INFO] new learning rate: 9.999999747378752e-05


In [89]:
epochs = 4
verbose = 1 #0
batch_size = 32
History = model.fit(train_X,
                        train_y,
                        batch_size=batch_size,   
                        epochs = epochs, 
                        validation_data = (validation_X,validation_y),
                        callbacks=callbacks,
                        verbose = verbose)

Epoch 1/4
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0193 - mae: 0.0193 - mape: 236.5138
Epoch 1: val_loss improved from None to 0.01845, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\E2-cp-0001-loss0.02.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 73s 25ms/step - loss: 0.0190 - mae: 0.0190 - mape: 232.7485 - val_loss: 0.0185 - val_mae: 0.0185 - val_mape: 8.4545
Epoch 2/4
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0179 - mae: 0.0179 - mape: 174.2052
Epoch 2: val_loss improved from 0.01845 to 0.01640, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\E2-cp-0002-loss0.02.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 62s 23ms/step - loss: 0.0175 - mae: 0.0175 - mape: 155.1966 - val_loss: 0.0164 - val_mae: 0.0164 - val_mape: 6.7442
Epoch 3/4
2651/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0163 - mae: 0.0163 - mape: 12.4215
Epoch 3: val_loss improved from 0.01640 to 0.01495, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\E2-cp-0003-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 63s 24ms/step - loss: 0.0159 - mae: 0.0159 - mape: 44.6346 - val_loss: 0.0149 - val_mae: 0.0149 - val_mape: 6.3391
Epoch 4/4
2653/2653 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0149 - mae: 0.0149 - mape: 33.5020
Epoch 4: val_loss improved from 0.01495 to 0.01449, saving model to C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\E2-cp-0004-loss0.01.h5


2653/2653 ━━━━━━━━━━━━━━━━━━━━ 63s 24ms/step - loss: 0.0146 - mae: 0.0146 - mape: 47.0993 - val_loss: 0.0145 - val_mae: 0.0145 - val_mape: 6.4107


In [92]:
model = load_model(r'C:\Users\hp\Downloads\MACHINE LEARNING LAB\Lab 12\E2-cp-0004-loss0.01.h5', compile=False)

y_pred_scaled   = model.predict(test_X)
y_pred          = scaler.inverse_transform(y_pred_scaled)
y_test_unscaled = scaler.inverse_transform(test_y)
# Mean Absolute Error (MAE)
MAE = np.mean(abs(y_pred - y_test_unscaled)) 
print('Mean Absolute Error (MAE): ' + str(np.round(MAE, 2)))

# Median Absolute Error (MedAE)
MEDAE = np.median(abs(y_pred - y_test_unscaled))
print('Median Absolute Error (MedAE): ' + str(np.round(MEDAE, 2)))

# Mean Squared Error (MSE)
MSE = np.square(np.subtract(y_pred, y_test_unscaled)).mean()
print('Mean Squared Error (MSE): ' + str(np.round(MSE, 2)))

# Root Mean Squarred Error (RMSE) 
RMSE = np.sqrt(np.mean(np.square(y_pred - y_test_unscaled)))
print('Root Mean Squared Error (RMSE): ' + str(np.round(RMSE, 2)))

# Mean Absolute Percentage Error (MAPE)
MAPE = np.mean((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Mean Absolute Percentage Error (MAPE): ' + str(np.round(MAPE, 2)) + ' %')

# Median Absolute Percentage Error (MDAPE)
MDAPE = np.median((np.abs(np.subtract(y_test_unscaled, y_pred)/ y_test_unscaled))) * 100
print('Median Absolute Percentage Error (MDAPE): ' + str(np.round(MDAPE, 2)) + ' %')

print('\n\ny_test_unscaled.shape= ',y_test_unscaled.shape)
print('y_pred.shape= ',y_pred.shape)

379/379 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step
Mean Absolute Error (MAE): 226.32
Median Absolute Error (MedAE): 184.8
Mean Squared Error (MSE): 84231.43
Root Mean Squared Error (RMSE): 290.23
Mean Absolute Percentage Error (MAPE): 1.55 %
Median Absolute Percentage Error (MDAPE): 1.27 %


y_test_unscaled.shape=  (12105, 1)
y_pred.shape=  (12105, 1)


## Final Conclusion
* In this lab, an LSTM (Long Short-Term Memory) neural network was developed for time-series forecasting.
* Sequential data was prepared and processed for use in the LSTM model.
* The network architecture was designed to capture temporal patterns in the data.
* Callback functions were utilized to monitor and manage the training process efficiently.
* The forecasting model was evaluated using various regression performance metrics to assess prediction accuracy.

